# FiD — Leveraging Passage Retrieval with Generative Models for Open Domain Question Answering (2020)

_Papers · Transformers & LLMs_

**Encode each retrieved passage on its own, then let the decoder fuse evidence across all of them at once.**

---

This notebook is a practice scaffold for the **FiD — Leveraging Passage Retrieval with Generative Models for Open Domain Question Answering (2020)** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np, matplotlib.pyplot as plt

## Reference implementation — PyTorch

In [ ]:
import torch, torch.nn as nn
import random, numpy as np

torch.manual_seed(0); np.random.seed(0); random.seed(0)

# --- 0. Sanity-check the worked example: predicted accuracy curve. ---
# Clue in 1 of N=6 random passages; C=8 answers; decoder fuses first k passages.
# Visible -> correct; else guess at 1/C.  E[acc] = k/6 + (1 - k/6)/8.
N, C = 6, 8
for k in range(1, 7):
    p = k / N
    print("k=%d  P(clue visible)=%.4f  E[acc]=%.4f" % (k, p, p + (1 - p) / C))
# k=1 ... E[acc]=0.2708     k=3 ... 0.5625     k=6 ... 1.0000  -> matches the run below


# --- 1. Toy data: a "retriever" returns N passages; the answer clue is in ONE. ---
L = 5                                  # tokens per passage
FILL  = list(range(0, 6))              # distractor filler tokens
SIGNAL = list(range(6, 6 + C))         # one signal token per answer class
V = 6 + C                              # vocab size
def make_example(rng):
    answer = rng.randrange(C)                                   # 0..C-1
    passages = [[rng.choice(FILL) for _ in range(L)] for _ in range(N)]
    pa = rng.randrange(N)                                       # which passage holds it
    passages[pa][rng.randrange(L)] = SIGNAL[answer]             # signal token = the answer
    return passages, answer

# --- 2. The model: tiny encoder + decoder composed with torch.nn. ---
D = 32
class FiD(nn.Module):
    def __init__(self):
        super().__init__()
        self.emb = nn.Embedding(V, D)
        self.pos = nn.Parameter(torch.randn(L, D) * 0.02)
        enc = nn.TransformerEncoderLayer(D, 4, 64, batch_first=True, dropout=0.0)
        self.encoder = nn.TransformerEncoder(enc, 2)            # SHARED across passages
        self.query = nn.Parameter(torch.randn(1, D) * 0.02)     # decoder seed token
        dec = nn.TransformerDecoderLayer(D, 4, 64, batch_first=True, dropout=0.0)
        self.decoder = nn.TransformerDecoder(dec, 2)
        self.head = nn.Linear(D, C)

    def encode_one(self, p):                                    # encode ONE passage alone
        x = self.emb(torch.tensor(p)).unsqueeze(0) + self.pos.unsqueeze(0)
        return self.encoder(x)                                  # self-attn over ONE passage

    def forward(self, passages, k_fuse):
        # FUSION-IN-DECODER: encode each passage independently, then CONCATENATE
        # all encodings and let the decoder attend over the union (Sec 3).
        mem = torch.cat([self.encode_one(p) for p in passages[:k_fuse]], dim=1)   # [1, k*L, D]
        out = self.decoder(self.query.unsqueeze(0), mem)        # decoder fuses the set
        return self.head(out[:, 0, :])                          # [1, C]

# --- 3. Train + evaluate at a given fuse count k. ---
def train_eval(k_fuse, seed=0, iters=4000):
    torch.manual_seed(seed); np.random.seed(seed)
    rng = random.Random(seed); model = FiD()
    opt = torch.optim.Adam(model.parameters(), lr=1.5e-3); lossf = nn.CrossEntropyLoss()
    model.train()
    for _ in range(iters):
        passages, ans = make_example(rng)
        loss = lossf(model(passages, k_fuse), torch.tensor([ans]))
        opt.zero_grad(); loss.backward(); opt.step()
    model.eval(); rng_e = random.Random(5000 + seed); correct = 0; Nev = 500
    with torch.no_grad():
        for _ in range(Nev):
            passages, ans = make_example(rng_e)
            correct += (model(passages, k_fuse).argmax(-1).item() == ans)
    return correct / Nev

print("\nAccuracy vs number of passages the decoder fuses (6 retrieved):")
for k in [1, 2, 3, 4, 5, 6]:
    print("  k_fuse=%d  accuracy=%.3f" % (k, train_eval(k)))

# --- 4. ABLATION: 6 retrieved, but decoder fuses only 1 (no fusion) vs all 6. ---
print("\nAblation  no-fusion (k=1): %.3f    full fusion (k=6): %.3f"
      % (train_eval(1), train_eval(6)))
# Typical single-seed run (CPU):
#   k_fuse=1 ~0.22   k_fuse=2 ~0.32   k_fuse=3 ~0.56   k_fuse=4 ~0.70   k_fuse=5 ~0.86   k_fuse=6 ~1.00
# Ablation: fusing 1 of 6 ~0.22  vs  fusing all 6 ~1.00.
# (Our small run, not the paper's reported numbers. Exact values vary by seed/hardware.)

## Visualize the data & results

_As the decoder fuses MORE of the retrieved passages, does answer accuracy rise when the evidence is spread across the passage set?_

In [ ]:
import torch, torch.nn as nn, random, numpy as np

N, C, L, D = 6, 8, 5, 32
FILL = list(range(0, 6)); SIGNAL = list(range(6, 6 + C)); V = 6 + C
def make_example(rng):
    ans = rng.randrange(C)
    ps = [[rng.choice(FILL) for _ in range(L)] for _ in range(N)]
    ps[rng.randrange(N)][rng.randrange(L)] = SIGNAL[ans]      # clue in ONE random passage
    return ps, ans

class FiD(nn.Module):
    def __init__(self):
        super().__init__()
        self.emb = nn.Embedding(V, D); self.pos = nn.Parameter(torch.randn(L, D) * 0.02)
        self.encoder = nn.TransformerEncoder(
            nn.TransformerEncoderLayer(D, 4, 64, batch_first=True, dropout=0.0), 2)
        self.query = nn.Parameter(torch.randn(1, D) * 0.02)
        self.decoder = nn.TransformerDecoder(
            nn.TransformerDecoderLayer(D, 4, 64, batch_first=True, dropout=0.0), 2)
        self.head = nn.Linear(D, C)
    def enc1(self, p):                                        # encode ONE passage alone
        return self.encoder(self.emb(torch.tensor(p)).unsqueeze(0) + self.pos.unsqueeze(0))
    def forward(self, ps, k):                                 # FUSION-IN-DECODER
        mem = torch.cat([self.enc1(p) for p in ps[:k]], dim=1)   # concat the encodings
        return self.head(self.decoder(self.query.unsqueeze(0), mem)[:, 0, :])

def train_eval(k, seed, iters=4000):
    torch.manual_seed(seed); np.random.seed(seed); rng = random.Random(seed)
    m = FiD(); opt = torch.optim.Adam(m.parameters(), lr=1.5e-3); lf = nn.CrossEntropyLoss()
    m.train()
    for _ in range(iters):
        ps, a = make_example(rng)
        loss = lf(m(ps, k), torch.tensor([a])); opt.zero_grad(); loss.backward(); opt.step()
    m.eval(); rng_e = random.Random(5000 + seed); ok = 0
    with torch.no_grad():
        for _ in range(500):
            ps, a = make_example(rng_e); ok += (m(ps, k).argmax(-1).item() == a)
    return ok / 500

for k in [1, 2, 3, 4, 5, 6]:
    accs = [train_eval(k, s) for s in (0, 1, 2)]
    print("k=%d  measured=%.3f  predicted=%.4f" % (k, sum(accs)/3, k/N + (1 - k/N)/C))
# k=1 measured=0.220 predicted=0.2708     k=4 measured=0.703 predicted=0.7083
# k=2 measured=0.320 predicted=0.4167     k=5 measured=0.860 predicted=0.8542
# k=3 measured=0.556 predicted=0.5625     k=6 measured=1.000 predicted=1.0000
# More fused passages -> spread-out evidence is found -> higher accuracy.
# Our small run, not the paper's number.

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** The fusion ablation. You have a working FiD reader. Six passages are retrieved, with the answer
            clue in one random passage of the set. You change the decoder to fuse only the FIRST passage
            (k_fuse = 1) instead of all six, everything else identical. What did you just remove, and
            what do you expect to happen to accuracy?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Locate the memory build: mem = torch.cat([encode(p) for p in passages[:k_fuse]], dim=1). Set k_fuse = 1. — _Now the decoder's cross-attention sees only passage 0's encoding, so it cannot pull a clue from passages 1&ndash;5._
- Reason about visibility: the clue is in a uniform random one of 6 passages, so with only passage 0 fused it is visible $1/6$ of the time; otherwise the model must guess among $C=8$ answers. — _Expected accuracy $= \tfrac{1}{6}\cdot 1 + \tfrac{5}{6}\cdot\tfrac{1}{8} \approx 0.27$ &mdash; near chance._
- Compare to full fusion ($k=6$): the clue is always visible, so accuracy approaches 1.0. — _Fusion across the whole retrieved set is what lets the decoder find evidence wherever it landed._

**Answer:** You removed the cross-passage fusion: with k_fuse = 1 the decoder attends to a
                 single passage, so it can answer only when the clue happens to sit in that one passage
                 ($\approx 1/6$ of the time) and otherwise guesses. Accuracy collapses from $\approx 1.0$ (fuse all
                 6) to $\approx 0.27$. This isolates the decoder fusion as the source of the gain &mdash; the same
                 reason the paper's accuracy rises with the number of passages (&sect;4, Figure 3).

</details>

**Problem 2.** The linear-cost argument. You have $N$ passages, each of length $\ell$ tokens. Compare the
            encoder self-attention cost of (A) gluing all passages into one input and encoding them together, versus
            (B) FiD's per-passage encoding. Which is linear in $N$, and why does that let FiD read 100 passages?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Recall self-attention cost scales like the square of the sequence length: length $T$ costs $\sim T^2$. — _Every token attends to every other token, so comparisons grow with $T^2$._
- Option A: one input of length $T = N\ell$. Cost $\sim (N\ell)^2 = N^2\ell^2$ &mdash; quadratic in $N$. — _All passages attend to all others, so doubling $N$ quadruples encoder cost._
- Option B (FiD): encode each length-$\ell$ passage alone, $N$ times. Cost $\sim N\cdot\ell^2$ &mdash; linear in $N$. — _Passages do not attend across each other in the encoder, so cost just adds up per passage._

**Answer:** Option A (encode together) is $N^2\ell^2$ &mdash; quadratic in the passage count. FiD's
                 Option B is $N\ell^2$ &mdash; linear, because each passage is encoded on its own and never
                 attends to the others. The decoder's cross-attention over the concatenation is also linear in $N$
                 and cheap because the answer is short. Linear cost is exactly what lets FiD scale to 100 passages
                 (&sect;3), where the naive quadratic model cannot.

</details>

**Problem 3.** In the worked example ($N=6$ passages, clue in a uniform random one, $C=8$ answers), the predicted
            accuracy was $\mathbb{E}[\text{acc}] = \tfrac{k}{6} + (1-\tfrac{k}{6})\tfrac{1}{8}$. Suppose instead
            the retriever returned $N=10$ passages (clue still in one random passage) and the decoder fused $k=4$.
            Compute the expected accuracy. What does this say about retrieving more passages than you fuse?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Probability the clue is visible in the first $k=4$ of $N=10$: $P = 4/10 = 0.4$. — _The clue is uniform over 10 passages; only the first 4 are fused._
- Expected accuracy $= 0.4\cdot 1 + (1 - 0.4)\cdot\tfrac{1}{8} = 0.4 + 0.6\cdot 0.125 = 0.4 + 0.075 = 0.475$. — _Visible &rarr; correct; not visible &rarr; guess at $1/8$._
- Compare: with $N=6$, $k=4$ gave $0.71$; with $N=10$, $k=4$ gives $0.475$. — _Spreading the clue over more retrieved passages while fusing the same number lowers the chance it is in the window._

**Answer:** $\mathbb{E}[\text{acc}] = 0.4 + 0.6\cdot 0.125 = 0.475$. Retrieving more passages only helps if
                 the decoder actually fuses them: with the clue spread over $N=10$ but only $k=4$ fused, the
                 evidence is outside the window more often, so accuracy drops to $0.475$ versus $0.71$ at $N=6$,
                 $k=4$. This is why FiD's payoff comes from fusing many passages in the decoder, not from
                 retrieving many and reading few &mdash; the whole point of the linear cost is that you can
                 afford to fuse them all.

</details>